# IndiaScored — Model Development

**Author:** Akshat Sarkar  
**Project:** IndiaScored — AI credit risk for India's thin-file borrowers

This notebook builds the model that IndiaScored serves. It runs end to end:

| Section | What happens |
|---|---|
| 1 | Why the training data has to be synthesised, and how |
| 2 | Generate a labelled thin-file applicant population |
| 3 | Inspect the population — marginals, separation, leakage checks |
| 4 | Feature engineering and the 70 / 10 / 20 split |
| 5 | Handle class imbalance with SMOTE, on the training fold only |
| 6 | Tune LightGBM with Optuna (TPE) against validation ROC-AUC |
| 7 | Calibrate the probabilities — a PD has to mean what it says |
| 8 | Evaluate: ranking, calibration and the cost of a threshold |
| 9 | Explain with SHAP, and sanity-check the directions |
| 10 | Translate PD into the credit decisions the API serves |
| 11 | Export the deployable bundle and verify it loads |

The heavy lifting lives in `backend/indiascored/training/`, imported below rather
than pasted in, so that the code the notebook trains with is byte-for-byte the code
the repository ships and tests.


## 0. Environment

```bash
cd backend
pip install -r requirements-training.txt
```


In [ ]:
import sys
from pathlib import Path

# Run from anywhere inside the repository.
BACKEND = Path.cwd().parent / 'backend' if Path.cwd().name == 'notebooks' else Path.cwd() / 'backend'
sys.path.insert(0, str(BACKEND))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 40)
plt.rcParams['figure.figsize'] = (11, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print('backend on path:', BACKEND)


## 1. Why synthesise the data

IndiaScored exists for people with **no bureau record**. That is exactly the
population no public dataset covers: if someone appears in a lending dataset,
they were already lent to, which means they were already scoreable. Training on
such a dataset would bake in the very exclusion the product is trying to undo.

So the training population is simulated from a **structural model**:

1. Each alternative signal is drawn from a plausible marginal distribution
   (Poisson SMS counts, Beta bill punctuality, regional location stability).
2. The signals combine into a latent log-odds of default through hand-set
   weights that encode domain priors — bill punctuality and the psychometric
   result matter most, SIM tenure least.
3. Non-linear interactions are layered on: chronically late recharges, a rural
   applicant with no cooperative standing, income below subsistence, a startup
   loan to a brand-new number. These are what make the problem worth a gradient
   booster rather than a logistic regression.
4. Gaussian noise is added so the signals are informative but never deterministic.
5. The intercept is solved numerically (Brent's method) so the realised default
   rate lands on a target of 20%.

> **Read the metrics accordingly.** They describe how well this pipeline recovers
> a known generating process. They are not a claim about the Indian credit market.
> What transfers to production is the pipeline — the features, the calibration, the
> explanation layer — not the AUC.


In [ ]:
from indiascored.training import SynthesisConfig, generate_dataset
from indiascored.training.synthesis import DEFAULT_WEIGHTS

for signal, weight in sorted(DEFAULT_WEIGHTS.items(), key=lambda kv: kv[1]):
    direction = 'lowers risk ' if weight < 0 else 'raises risk '
    print(f'{signal:22s} {weight:+.2f}   {direction}')


## 2. Generate the population


In [ ]:
config = SynthesisConfig(
    n_applicants=12_000,
    target_default_rate=0.20,
    random_seed=20250919,
)

applicants = generate_dataset(config)

print('rows:', len(applicants))
print('realised default rate:', f"{applicants['defaulted'].mean():.2%}")
applicants.head()


In [ ]:
DATA_DIR = BACKEND.parent / 'data'
DATA_DIR.mkdir(exist_ok=True)

csv_path = DATA_DIR / 'indiascored_applicants.csv'
applicants.to_csv(csv_path, index=False)
print('written:', csv_path)


## 3. Inspect the population

Three things to confirm before modelling: the marginals are plausible, the
classes are genuinely separable, and nothing has leaked.


In [ ]:
applicants.describe(include='all').T


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))

panels = [
    ('sms_count', 'Transactional SMS per month'),
    ('bill_on_time_ratio', 'Utility bill punctuality'),
    ('sim_tenure', 'SIM tenure (months)'),
    ('coop_score', 'Cooperative / SHG standing'),
    ('psychometric_score', 'Psychometric result'),
    ('income_signal', 'Inferred income stability'),
]

for ax, (column, title) in zip(axes.ravel(), panels):
    for label, subset in applicants.groupby('defaulted'):
        ax.hist(subset[column], bins=30, alpha=0.55,
                label='defaulted' if label else 'repaid', density=True)
    ax.set_title(title, fontsize=10)

axes[0][0].legend()
fig.suptitle('Signal distributions by outcome', fontsize=12)
fig.tight_layout()
plt.show()


In [ ]:
# Class separation: how far apart are the group means, in pooled standard deviations?
numeric = ['sms_count', 'bill_on_time_ratio', 'recharge_freq', 'sim_tenure',
           'location_stability', 'income_signal', 'coop_score',
           'land_verified', 'psychometric_score', 'loan_amount_requested']

repaid = applicants[applicants.defaulted == 0]
defaulted = applicants[applicants.defaulted == 1]

separation = pd.DataFrame({
    'repaid_mean': repaid[numeric].mean(),
    'defaulted_mean': defaulted[numeric].mean(),
})
pooled_sd = applicants[numeric].std()
separation['cohens_d'] = (separation.defaulted_mean - separation.repaid_mean) / pooled_sd
separation.sort_values('cohens_d', key=abs, ascending=False).round(3)


In [ ]:
# Leakage check: the true PD used to draw the label must never reach the model.
LEAKY = {'true_pd', 'defaulted', 'applicant_id'}
from indiascored.training.pipeline import CATEGORICAL_FEATURES, NUMERIC_FEATURES

model_inputs = set(NUMERIC_FEATURES) | set(CATEGORICAL_FEATURES)
assert not (model_inputs & LEAKY), 'a label-derived column reached the feature set'
print('model sees', len(model_inputs), 'columns; none of them leak the label')


## 4. Feature engineering and the split

`derive_features` is imported from the serving code, not reimplemented here.
That is deliberate: train/serve skew in a credit model shows up as silently
wrong decisions, and the only reliable defence is one implementation.

It adds three columns:

- `loan_amount_log` — loan size is heavy-tailed, so the model reads it on a log scale
- `sms_norm` — SMS volume against a **frozen** reference ceiling, never a batch max,
  so a single-applicant request scores identically alone and in a batch
- `sim_tenure_years` — tenure on a human scale

The split is stratified 70 / 10 / 20. The validation fold does double duty: Optuna
selects on it, and the probability calibrator is fitted on it.


In [ ]:
from indiascored.scoring.features import derive_features
from indiascored.training.pipeline import TrainingConfig, split_dataset, build_preprocessor

training_config = TrainingConfig(random_state=20250919, optuna_trials=40)

X_train, X_val, X_test, y_train, y_val, y_test = split_dataset(applicants, training_config)

for name, y in [('train', y_train), ('validation', y_val), ('test', y_test)]:
    print(f'{name:11s} n={len(y):6,d}   default rate={y.mean():.2%}')


In [ ]:
preprocessor = build_preprocessor()

X_train_enc = preprocessor.fit_transform(X_train)
X_val_enc = preprocessor.transform(X_val)
X_test_enc = preprocessor.transform(X_test)

feature_names = [str(name) for name in preprocessor.get_feature_names_out()]
print('encoded width:', X_train_enc.shape[1])
feature_names


## 5. Class imbalance

One in five applicants defaults, so a model that predicts "nobody defaults"
is already 80% accurate and completely useless. SMOTE synthesises minority-class
neighbours to balance the training fold.

**Only the training fold.** Resampling before the split would place synthetic
rows interpolated from test applicants into the training set, and the test
score would be a measure of nothing.


In [ ]:
from indiascored.training.pipeline import resample

X_train_res, y_train_res = resample(X_train_enc, y_train, training_config)

print(f'before SMOTE: {len(y_train):,} rows, {y_train.mean():.1%} defaults')
print(f'after  SMOTE: {len(y_train_res):,} rows, {np.mean(y_train_res):.1%} defaults')
print(f'validation and test folds untouched: {len(y_val):,} / {len(y_test):,}')


## 6. Hyper-parameter search

LightGBM handles the mixed tabular feature space and the interactions built into
the generating process. Optuna's TPE sampler searches the space, scoring each
trial on **validation ROC-AUC** — a model fitted on resampled data and selected
on resampled data would be tuned for a population that does not exist.


In [ ]:
from indiascored.training.pipeline import search_hyperparameters

best_params = search_hyperparameters(X_train_res, y_train_res, X_val_enc, y_val, training_config)

for key, value in best_params.items():
    print(f'{key:20s} {value}')


In [ ]:
import lightgbm as lgb

booster = lgb.LGBMClassifier(
    objective='binary',
    random_state=training_config.random_state,
    n_jobs=-1,
    verbose=-1,
    **best_params,
)
booster.fit(X_train_res, y_train_res)
print('trees:', booster.n_estimators_)


## 7. Probability calibration

A ranking model is not enough here. The API turns the probability into a 300-900
IndiaScore, a risk grade and a sanctioned rupee amount, so a PD of 0.15 has to
mean that roughly fifteen applicants in a hundred like this one will default.

SMOTE deliberately distorted the base rate in training, so the raw booster's
probabilities are inflated. Isotonic regression, fitted on the untouched
validation fold, maps them back onto reality.


In [ ]:
from sklearn.calibration import CalibratedClassifierCV

calibrated = CalibratedClassifierCV(
    booster,
    method=training_config.calibration_method,
    cv=training_config.calibration_folds,
)
calibrated.fit(X_val_enc, y_val)

raw_test_pd = booster.predict_proba(X_test_enc)[:, 1]
cal_test_pd = calibrated.predict_proba(X_test_enc)[:, 1]

print(f'observed default rate  {y_test.mean():.3f}')
print(f'mean raw PD            {raw_test_pd.mean():.3f}')
print(f'mean calibrated PD     {cal_test_pd.mean():.3f}')


In [ ]:
from sklearn.calibration import calibration_curve

fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.5))

left.plot([0, 1], [0, 1], 'k--', lw=1, label='perfect calibration')
for label, probabilities in [('raw LightGBM', raw_test_pd), ('isotonic-calibrated', cal_test_pd)]:
    observed, predicted = calibration_curve(y_test, probabilities, n_bins=10, strategy='quantile')
    left.plot(predicted, observed, 'o-', label=label)
left.set_xlabel('predicted probability of default')
left.set_ylabel('observed default rate')
left.set_title('Reliability on the test fold')
left.legend()

right.hist(cal_test_pd, bins=40)
right.axvline(y_test.mean(), color='k', ls='--', label='observed default rate')
right.set_xlabel('calibrated PD')
right.set_title('Where the test population sits')
right.legend()

fig.tight_layout()
plt.show()


## 8. Evaluation

Four numbers, each answering a different question:

- **ROC-AUC** — can the model rank a defaulter above a repayer?
- **PR-AUC** — does it find defaulters, given they are only a fifth of the book?
  The no-skill floor is the default rate itself, not 0.5.
- **Brier** — are the probabilities honest? This is the one that licenses the
  PD to become a rupee amount.
- **Recall on defaults** — of the applicants who did default, how many were caught?


In [ ]:
from indiascored.training.pipeline import evaluate

scores = pd.DataFrame({
    'raw booster (test)': evaluate(y_test, raw_test_pd),
    'calibrated (test)': evaluate(y_test, cal_test_pd),
}).T
scores.round(4)


In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.5))

fpr, tpr, _ = roc_curve(y_test, cal_test_pd)
left.plot(fpr, tpr, label=f"calibrated (AUC={scores.loc['calibrated (test)', 'roc_auc']:.3f})")
left.plot([0, 1], [0, 1], 'k--', lw=1)
left.set_xlabel('false positive rate'); left.set_ylabel('true positive rate')
left.set_title('ROC'); left.legend()

precision, recall, _ = precision_recall_curve(y_test, cal_test_pd)
right.plot(recall, precision, label=f"PR-AUC={scores.loc['calibrated (test)', 'pr_auc']:.3f}")
right.axhline(y_test.mean(), color='k', ls='--', lw=1, label=f'no skill ({y_test.mean():.2f})')
right.set_xlabel('recall'); right.set_ylabel('precision')
right.set_title('Precision-Recall'); right.legend()

fig.tight_layout()
plt.show()


In [ ]:
# A threshold is a business decision, not a modelling one. This is what each costs.
from sklearn.metrics import precision_score, recall_score, f1_score

rows = []
for threshold in np.arange(0.10, 0.65, 0.05):
    flagged = (cal_test_pd >= threshold).astype(int)
    rows.append({
        'threshold': round(threshold, 2),
        'share_declined': flagged.mean(),
        'precision': precision_score(y_test, flagged, zero_division=0),
        'recall': recall_score(y_test, flagged, zero_division=0),
        'f1': f1_score(y_test, flagged, zero_division=0),
        'defaults_missed': int(((flagged == 0) & (y_test.values == 1)).sum()),
        'good_applicants_turned_away': int(((flagged == 1) & (y_test.values == 0)).sum()),
    })

pd.DataFrame(rows).set_index('threshold').round(3)


## 9. Explanation

A rejection an officer cannot explain is a rejection the applicant cannot
appeal, and in India it is also a regulatory problem. SHAP attributes every
individual decision back to the signals that drove it.

The explainer is built on the **raw booster**: the calibrator is a monotone
transform applied after the trees, so it reorders nothing and the attributions
hold for the calibrated output too.


In [ ]:
import shap

explainer = shap.TreeExplainer(booster)
shap_values = explainer.shap_values(X_test_enc)
if isinstance(shap_values, list):
    shap_values = shap_values[1]
shap_values = np.asarray(shap_values)
print('attribution matrix:', shap_values.shape)


In [ ]:
importance = (
    pd.Series(np.abs(shap_values).mean(axis=0), index=feature_names)
    .sort_values(ascending=False)
)

top = importance.head(14)[::-1]
plt.figure(figsize=(9, 6))
plt.barh(top.index, top.values)
plt.xlabel('mean |SHAP value|')
plt.title('What drives an IndiaScored decision')
plt.tight_layout()
plt.show()

importance.head(10).round(4)


In [ ]:
# Direction check: does the model agree with the credit priors it was built on?
# A negative correlation between a feature's value and its SHAP contribution
# means that more of the signal pushes the probability of default down.
EXPECTED_DIRECTION = {
    'num__bill_on_time_ratio': 'reduces',
    'num__psychometric_score': 'reduces',
    'num__coop_score': 'reduces',
    'num__income_signal': 'reduces',
    'num__land_verified': 'reduces',
    'num__location_stability': 'reduces',
    'num__recharge_freq': 'reduces',
    'num__sim_tenure': 'reduces',
}

rows = []
for feature, expected in EXPECTED_DIRECTION.items():
    column = feature_names.index(feature)
    correlation = np.corrcoef(X_test_enc[:, column], shap_values[:, column])[0, 1]
    observed = 'reduces' if correlation < 0 else 'raises'
    rows.append({
        'feature': feature,
        'corr(value, shap)': round(correlation, 3),
        'observed': observed,
        'expected': expected,
        'agrees': observed == expected,
    })

directions = pd.DataFrame(rows).set_index('feature')
assert directions.agrees.all(), directions[~directions.agrees]
directions


In [ ]:
# Loan size is carried by two collinear columns, so the pair is checked together
# rather than individually: a booster is free to load the signal onto either one.
size_columns = [feature_names.index('num__loan_amount_requested'),
                feature_names.index('num__loan_amount_log')]

combined_value = X_test_enc[:, size_columns].mean(axis=1)
combined_shap = shap_values[:, size_columns].sum(axis=1)
correlation = np.corrcoef(combined_value, combined_shap)[0, 1]

print(f'corr(loan size, combined SHAP) = {correlation:+.3f}')
print('a larger ask', 'RAISES' if correlation > 0 else 'LOWERS', 'the modelled risk')
print()
print('Mean calibrated PD by requested-amount quintile:')
quintiles = pd.qcut(X_test['loan_amount_requested'], 5)
print(pd.Series(cal_test_pd, index=X_test.index).groupby(quintiles, observed=True).mean().round(3))


In [ ]:
# One applicant, explained the way the dashboard explains them.
row = 0
contributions = (
    pd.Series(shap_values[row], index=feature_names)
    .sort_values(key=abs, ascending=False)
    .head(8)[::-1]
)

plt.figure(figsize=(9, 4.5))
plt.barh(contributions.index, contributions.values,
         color=['tab:red' if v > 0 else 'tab:green' for v in contributions.values])
plt.axvline(0, color='k', lw=1)
plt.xlabel('SHAP contribution to log-odds of default  (red raises risk)')
plt.title(f'Applicant #{row}: calibrated PD = {cal_test_pd[row]:.3f}, actual outcome = {y_test.iloc[row]}')
plt.tight_layout()
plt.show()


## 10. From probability to decision

The credit policy lives in `indiascored.scoring.grading` and is pure maths, so
it is unit-tested independently of the model. Applying it here shows what the
book would actually look like.


In [ ]:
from indiascored.scoring.grading import grade_pd, sanctionable_amount

book = pd.DataFrame({
    'pd': cal_test_pd,
    'defaulted': y_test.values,
    'requested': X_test['loan_amount_requested'].values,
})

graded = book.pd.apply(grade_pd)
book['grade'] = [g.grade for g in graded]
book['india_score'] = [g.india_score for g in graded]
book['apr'] = [g.indicative_apr for g in graded]
book['sanctioned'] = [sanctionable_amount(r, g) for r, g in zip(book.requested, book.grade)]

by_grade = book.groupby('grade').agg(
    applicants=('pd', 'size'),
    mean_pd=('pd', 'mean'),
    observed_default_rate=('defaulted', 'mean'),
    mean_score=('india_score', 'mean'),
    apr=('apr', 'first'),
    total_sanctioned=('sanctioned', 'sum'),
).reindex(['A+', 'A', 'B', 'C', 'D']).dropna(how='all')

by_grade.round(3)


The column that matters is `observed_default_rate` against `mean_pd`. If the two
track each other down the ladder, the grades mean what the policy says they mean:
an A+ book really does default less often than a C book.


In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.5))

ladder = by_grade.index.tolist()
position = np.arange(len(ladder))
left.bar(position - 0.2, by_grade.mean_pd, width=0.4, label='predicted PD')
left.bar(position + 0.2, by_grade.observed_default_rate, width=0.4, label='observed default rate')
left.set_xticks(position); left.set_xticklabels(ladder)
left.set_title('Does each grade default as often as it claims?')
left.legend()

right.hist(book.india_score, bins=40)
right.set_xlabel('IndiaScore'); right.set_title('Score distribution across the test book')

fig.tight_layout()
plt.show()


## 11. Export the bundle

Everything the API needs goes into one file: preprocessor, raw booster,
calibrated classifier, SHAP explainer, encoded feature names and the metrics
this run produced. Deploying a retrained model is then a file drop — the
service reads its path from `MODEL_BUNDLE_PATH` and nothing in the API changes.

`train_bundle` re-runs the whole pipeline as one call, so the artifact can never
drift from the steps above through a cell run out of order.


In [ ]:
from indiascored.training import train_bundle

BUNDLE_PATH = BACKEND / 'artifacts' / 'indiascored_pipeline_bundle.pkl'

bundle = train_bundle(applicants, training_config, output_path=BUNDLE_PATH)

print('written:', BUNDLE_PATH)
print('size:', f'{BUNDLE_PATH.stat().st_size / 1e6:.1f} MB')
pd.DataFrame(bundle['metrics']).T.round(4)


In [ ]:
# Verify the artifact through the same code path the API uses.
from indiascored.scoring.bundle import load_bundle
from indiascored.scoring.engine import ScoringEngine

engine = ScoringEngine(load_bundle(BUNDLE_PATH))

profiles = {
    'strong rural farmer': dict(
        user_type='smartphone', region='rural', age_group='31-50', sms_count=32,
        bill_on_time_ratio=0.94, recharge_pattern='always_on_time', recharge_freq=1.0,
        sim_tenure=96, location_stability=0.88, income_signal=0.90, coop_score=86,
        land_verified=1, psychometric_score=0.84, loan_amount_requested=150_000,
        loan_category='farmer'),
    'middling urban applicant': dict(
        user_type='smartphone', region='urban', age_group='18-30', sms_count=22,
        bill_on_time_ratio=0.60, recharge_pattern='sometimes_late', recharge_freq=0.5,
        sim_tenure=24, location_stability=0.62, income_signal=0.50, coop_score=52,
        land_verified=0, psychometric_score=0.52, loan_amount_requested=200_000,
        loan_category='personal'),
    'weak thin-file applicant': dict(
        user_type='feature_phone', region='rural', age_group='18-30', sms_count=4,
        bill_on_time_ratio=0.18, recharge_pattern='often_late', recharge_freq=0.2,
        sim_tenure=3, location_stability=0.32, income_signal=0.15, coop_score=24,
        land_verified=0, psychometric_score=0.22, loan_amount_requested=450_000,
        loan_category='startup'),
}

for name, profile in profiles.items():
    card = engine.score(profile).as_dict()
    print(f"{name:26s} PD={card['probability_of_default']:.3f}  "
          f"score={card['india_score']:3d}  grade={card['grade']:2s}  "
          f"sanction={card['sanctioned_amount']:>8,d}  {card['decision']}")
    for driver in card['drivers'][:3]:
        arrow = 'raises' if driver['contribution'] > 0 else 'lowers'
        print(f"      {driver['feature']:34s} {arrow} risk ({driver['contribution']:+.3f})")
    print()


## What comes next

- **Sequential models (LSTM / temporal transformers)** over recharge and bill
  time series, so the *trajectory* of a thin-file applicant is read, not just
  their current snapshot.
- **Generative psychometric item banks**, so no two applicants see the same
  questions and the assessment cannot be coached.
- **Kafka-streamed telecom events** feeding scores continuously, turning the
  score from a point-in-time verdict into a live risk signal.
- **Fairness auditing** across region, gender and age bands before any real
  lending decision rides on this model — the synthetic population cannot
  surface the disparate impact a real one would.

---

*IndiaScored — Akshat Sarkar*
